# Experiments – Sebastien

Follow-up to `experiments_edouard.ipynb`. The CPU study in `experiments/sebastien/` suggests the held-out
sessions that fail are the ones whose **combination of conditions** (facing side, eye size, pupil polarity) is
missing from training (110535, 171101), and that lighting augmentation beyond hflip helps on them.
This notebook tests that in DLC: hist_eq + hflip (Edouard's best), with and without lighting-altered
copies of the training frames (`src/photometric.py`).

Not run yet (needs a GPU, ~35 min per shuffle). Needs `dlc/pupil-dani-2026-11-13/` and `data/labeled-data/` in
the canonical layout (`python -m src.canonicalize <lab export> data/labeled-data` if starting from the lab's export).

The copies live in a separate project, `dlc/pupil-photometric/`: DLC merges every labeled-data folder listed
in config.yaml into each new training set and stores splits as row numbers into that merge, so adding folders
to the original project would shift the splits of shuffles 1–17.

In [ ]:
from pathlib import Path

from src.data import load_labels, session_conditions, unseen_condition_sessions
from src.metrics import compare_runs, evaluate_run, find_predictions, load_predictions
from src.photometric import create_photometric_project
from src.shuffles import DlcProject

ORIGINAL = Path("dlc") / "pupil-dani-2026-11-13"
PHOTOMETRIC = Path("dlc") / "pupil-photometric"
COPIES = 2  # lighting-altered copies per training frame

if not (PHOTOMETRIC / "config.yaml").exists():
    create_photometric_project(ORIGINAL, PHOTOMETRIC, copies=COPIES, seed=0)
project = DlcProject(PHOTOMETRIC)
print(project.frame_counts().to_string())

## Runs

Pairs that differ only by the copies. With copies the training set is 3x larger, so the copy runs use a third of
the epochs and of the learning-rate milestones: same number of gradient steps, same schedule.

Inference uses no hist_eq: DLC applies it with probability 0.5 at inference too, which made evaluations random.
Shuffle 1 is therefore also a deterministic re-run of Edouard's sh13.

| Shuffle | Held out | Training frames | Epochs | LR drops |
|---|---|---|---|---|
| 1 | 110535 (unseen combination) | originals | 100 | 90, 120 |
| 2 | 110535 | originals + copies | 33 | 30, 40 |
| 3 | 171101 (unseen combination) | originals | 100 | 90, 120 |
| 4 | 171101 | originals + copies | 33 | 30, 40 |
| 5 | 135059 (control) | originals | 100 | 90, 120 |
| 6 | 135059 | originals + copies | 33 | 30, 40 |

In [ ]:
TRAIN = {"hist_eq": True, "hflip": 0.5}
INFERENCE = {"hist_eq": False}

RUNS: dict[int, tuple[str, bool]] = {  # shuffle: (held-out session, with copies)
    1: ("110535", False),
    2: ("110535", True),
    3: ("171101", False),
    4: ("171101", True),
    5: ("135059", False),
    6: ("135059", True),
}


def schedule(with_copies: bool) -> tuple[int, list[int]]:
    """Epochs and learning-rate milestones giving every run the same number of gradient steps."""
    scale = 1 + COPIES if with_copies else 1
    return round(100 / scale), [round(90 / scale), round(120 / scale)]


for shuffle, (test_session, with_copies) in RUNS.items():
    project.create_shuffle(shuffle, test_session, with_copies)
    epochs, milestones = schedule(with_copies)
    project.configure_shuffle(shuffle, TRAIN, INFERENCE, epochs, milestones)

In [ ]:
for shuffle, (_, with_copies) in RUNS.items():
    project.train_shuffle(shuffle, schedule(with_copies)[0])

In [ ]:
for shuffle, (_, with_copies) in RUNS.items():
    project.evaluate_shuffle(shuffle, schedule(with_copies)[0])

## Comparison

Same scores as Edouard's table (`src/metrics.py`). Sessions whose combination of conditions no other session
has are flagged: an average over all sessions hides their errors.

In [ ]:
labels = load_labels(Path("data") / "labeled-data")
unseen = unseen_condition_sessions(session_conditions(labels))
print("unseen combinations:", unseen)

runs = {}
for shuffle, (test_session, with_copies) in RUNS.items():
    epochs = schedule(with_copies)[0]
    name = f"sh{shuffle} {test_session} {'+copies' if with_copies else 'originals'}"
    predictions = load_predictions(find_predictions(PHOTOMETRIC, shuffle, epochs))
    runs[name] = evaluate_run(predictions, labels, [test_session])

comparison = compare_runs(runs)
comparison["unseen_combination"] = comparison["test_sessions"].isin(unseen)
comparison.round(3)